# LowResPT — Combined Poster Figure

One figure, three blocks. Two independently editable checkpoints drive it:
`RECON_CKPT` for the left block, `PROBE_CKPT` for **both** redshift panels.

- **Left** — 2×2 masked (MAE-consistent) reconstruction examples: Original (grey solid)
  vs Reconstructed (blue dashed) with observed-frame emission-line markers. Protocol
  mirrors `recon.ipynb` Section B.
- **Middle** — zero-shot k-NN redshift probe (frozen encoder, concat patch tokens).
- **Right** — few-shot linear redshift probe (encoder fine-tuned end-to-end).

Middle/right logic mirrors `linear_probe_redshift.ipynb`; styling mirrors the poster
cells of both source notebooks.

In [ ]:
import os, sys, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib as mpl
import matplotlib.pyplot as plt

# This notebook lives in the image-encoder tree, but the spectrum model/data live in
# LowResPT and its outputs/ checkpoints are globbed relative to the cwd, so point both
# sys.path and the working directory there.
LOWRESPT_DIR = "/home/yacheng/ssl_outthere/encoder_spectrum/LowResPT"
os.chdir(LOWRESPT_DIR)
sys.path.insert(0, LOWRESPT_DIR)

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error

from model.low_res_pt import LowResPT
from data.dataset import LowResDataset
from data.datamodule import LowResDataModule

FITS   = "/home/yacheng/ssl_outthere/data/spectrum/DJA_spectra_v4.5.fits"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def best_ckpt(name, version):
    # Best (lowest val_hid_loss) checkpoint for a run name / version.
    return min(Path("outputs").glob(f"{name}/version_{version}/checkpoints/*.ckpt"),
               key=lambda p: float(p.stem.split("val_hid_loss=")[-1]))

# ---- Per-part checkpoints: EDIT ME. Two independent models drive the figure ----
# RECON_CKPT -> left reconstruction block; PROBE_CKPT -> BOTH redshift panels
# (zero-shot k-NN and few-shot linear share one checkpoint). Both default to the
# same run; change either (name, version) independently later.
RECON_CKPT = best_ckpt("low_res_pt_1_2_micron_noz_cut", 0)  # left  · reconstruction
PROBE_CKPT = best_ckpt("low_res_pt_1_2_micron_noz_cut_tokenweight", 0)  # mid+right · redshift probes
print(f"device={DEVICE}")
print("RECON_CKPT:", RECON_CKPT)
print("PROBE_CKPT:", PROBE_CKPT)

POSTER_DIR = "/home/yacheng/ssl_outthere/poster_figs"
os.makedirs(POSTER_DIR, exist_ok=True)

# Rest-frame emission lines (µm), sorted by wavelength.
EMISSION_LINES = {
    r'Ly$\alpha$':          0.1216,
    '[OII]':                0.3727,
    r'H$\beta$':            0.4861,
    '[OIII]':               0.5007,
    r'H$\alpha$':           0.6563,
    '[SII]':                0.6724,
    r'Pa$\zeta$':           0.9229,
    r'Pa$\epsilon$+[SIII]': 0.9539,
    r'Pa$\delta$':          1.0049,
    r'Pa$\gamma$':          1.0938,
}

## Left block — masked reconstruction examples

Build the reconstruction val split exactly as in `recon.ipynb` (Section B, masked /
MAE-consistent) and pick four hand-chosen example objects for the 2×2 grid.

In [ ]:
# ---- reconstruction config (mirrors recon.ipynb) ----
USE_JANSKY = False      # dataset flux units (must match how the model was trained)
EDGE_TRIM  = 2          # drop this many low-overlap edge pixels per end
BLOCK_K    = 3          # masked-recon block size (P=4/S=2 overlap -> 3)

@torch.no_grad()
def reconstruct(model, batch, masked=True, block_k=BLOCK_K):
    # Reconstruct patch tokens, overlap-averaged into a pixel spectrum (masked /
    # MAE-consistent protocol). Returns flux_norm / recon_norm on the shared pixel grid.
    flux  = batch["flux"].to(DEVICE)
    wave  = batch["wavelength"].to(DEVICE)
    vmask = batch["valid_mask"].to(DEVICE)
    err   = batch["err"].to(DEVICE)
    z     = batch["redshift"]

    R = model.reconstruct(flux, wave, vmask, masked=masked, block_k=block_k)
    recon_pat     = R["recon_patches"]
    patches       = R["target"]
    flux_norm     = R["flux_norm"]
    valid_patches = R["valid_patches"]
    L_used        = R["L_used"]
    mean, std     = R["stats"][:, :1], R["stats"][:, 1:]
    B, N, P       = patches.shape
    S             = model.hparams.stride

    # Per-pixel err propagated into normalised-flux space (same stretch as flux_norm),
    # for plotting yerr on the Original curve. Bad-err pixels (<= 0 / non-finite) -> NaN
    # so plots skip them.
    _, _, sd, scale = model.data_stretch(flux, vmask)
    err_norm = model.err_stretch(flux, err, scale, sd)
    err_ok   = vmask & torch.isfinite(err) & (err > 0)
    err_norm = torch.where(err_ok, err_norm, torch.full_like(err_norm, float("nan")))

    # Overlap-average FULLY-valid tokens only; a kept pixel is then always valid.
    token_full_valid = valid_patches.all(dim=-1)
    recon_pix = torch.zeros(B, L_used, device=DEVICE)
    count     = torch.zeros(B, L_used, device=DEVICE)
    for t in range(N):
        s = t * S
        w = token_full_valid[:, t].to(recon_pat.dtype).unsqueeze(-1)
        recon_pix[:, s:s+P] += recon_pat[:, t, :] * w
        count[:, s:s+P]     += w
    recon_mask = count > 0
    recon_pix  = recon_pix / count.clamp(min=1)

    return dict(
        wave       = wave[:, :L_used].cpu(),
        recon_mask = recon_mask.cpu(),
        flux_norm  = flux_norm[:, :L_used].cpu(),
        recon_norm = recon_pix.cpu(),
        err_norm   = err_norm[:, :L_used].cpu(),
        redshift   = z.cpu() if isinstance(z, torch.Tensor) else torch.as_tensor(z),
        N=N, P=P, S=S, L_used=L_used,
    )

# ---- reconstruction model (left block) + val split (matches recon.ipynb) ----
model_recon = LowResPT.load_from_checkpoint(RECON_CKPT, map_location=DEVICE).eval().to(DEVICE)
wl_min, wl_max = model_recon.hparams["wl_ref_min"], model_recon.hparams["wl_ref_max"]
print(f"recon model: patch={model_recon.hparams.patch_size} stride={model_recon.hparams.stride} "
      f"wl=[{wl_min}, {wl_max}]")

dm = LowResDataModule(fits_path=FITS, batch_size=256, num_workers=0,
                      min_sn50=1.0, min_redshift=0.0, use_jansky=USE_JANSKY,
                      wl_ref_min=wl_min, wl_ref_max=wl_max, frac_valid_pix=0.5)
dm.setup()
batch  = next(iter(dm.val_dataloader()))
R_mask = reconstruct(model_recon, batch, masked=True, block_k=BLOCK_K)
print(f"recon batch: {batch['flux'].shape}  N={R_mask['N']} (P={R_mask['P']}, S={R_mask['S']})")

## Middle / right blocks — redshift probes

Frozen zero-shot k-NN (concat patch tokens) vs an end-to-end fine-tuned linear head,
on a shared group-aware 50/50 split. Mirrors `linear_probe_redshift.ipynb`.

In [ ]:
# ---- redshift-probe config (mirrors linear_probe_redshift.ipynb) ----
MIN_SN50, MIN_REDSHIFT, MAX_REDSHIFT = 0, 1, 3
FRAC_VALID_PIX = 0.9
POOL       = "concat"     # both probes read out the flattened token sequence
SPLIT_FRAC = 0.5
KNN_K      = 5
SEED       = 42

# probes and metrics come from eval/, the same module the optuna sweep scores with
from eval.probes import metrics, knn_zeroshot, lasso_probe

def _pool(raw, tvm, pool):
    vm3   = tvm.float().unsqueeze(-1)
    tok   = raw * vm3
    mean_ = tok.sum(1) / vm3.sum(1).clamp(min=1)
    max_  = torch.nan_to_num(raw.masked_fill(tvm.unsqueeze(-1) == 0, float("-inf")).max(1).values, neginf=0.0)
    if pool == "concat":  return tok.reshape(tok.shape[0], -1)
    if pool == "mean":    return mean_
    if pool == "max":     return max_
    if pool == "meanmax": return torch.cat([mean_, max_], dim=-1)
    raise ValueError(pool)

@torch.no_grad()
def compute_embeddings(enc, dataset, pool="concat", batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0,
                        collate_fn=LowResDataModule._pad_collate)
    Xs, ys = [], []
    for b in loader:
        out = enc.compute_embedding_from_raw_spectrum(
            b["flux"].to(DEVICE), b["wavelength"].to(DEVICE), b["valid_mask"].to(DEVICE))
        Xs.append(_pool(out["patch_token"], out["token_valid_mask"], pool).cpu().numpy())
        ys.append(b["redshift"].numpy())
    return np.concatenate(Xs), np.concatenate(ys)

# ---- probe model (mid + right blocks; shared by zero-shot and linear probe) ----
model_probe = LowResPT.load_from_checkpoint(PROBE_CKPT, map_location=DEVICE).eval().to(DEVICE)

# ---- dataset, embeddings, shared group-aware split, both probes ----
ds_z = LowResDataset(FITS, min_sn50=MIN_SN50, min_redshift=MIN_REDSHIFT,
                     max_redshift=MAX_REDSHIFT, frac_valid_pix=FRAC_VALID_PIX)
X, y = compute_embeddings(model_probe, ds_z, pool=POOL)
gss  = GroupShuffleSplit(n_splits=1, train_size=SPLIT_FRAC, random_state=SEED)
tr, va = next(gss.split(X, y, groups=ds_z.objid))
X_tr, X_va, y_tr, y_va = X[tr], X[va], y[tr], y[va]
assert len(set(ds_z.objid[tr]) & set(ds_z.objid[va])) == 0, "objid leaked across split"
print(f"redshift probe: X={X.shape}  train={len(tr)}  val={len(va)}  z in ({MIN_REDSHIFT},{MAX_REDSHIFT}]")

pred_zs = knn_zeroshot(X_tr, y_tr, X_va, KNN_K)
# L1 on the FROZEN embeddings, at the same fixed penalty the optuna sweep scores
# with (optuna/configs/base.yaml). Selecting alpha per-eval on a holdout of the
# train half was measured and rejected: the holdout is too small and lands on
# heavy regularisation (sNMAD 0.0703 against the 0.0466 the same data supports).
LASSO_ALPHA = 5.0e-4
pred_lin, lin_info = lasso_probe(X_tr, y_tr, X_va, alpha=LASSO_ALPHA)
print(f"lasso alpha={lin_info['alpha']:.4g} [{lin_info['edge']}] "
      f"active={lin_info['n_active']}/{X.shape[1]}")
for tag, p in [("zero-shot kNN", pred_zs), ("linear probe (L1)", pred_lin)]:
    m = metrics(y_va, p)
    print(f"{tag:22s} R2={m['r2']:.3f}  sNMAD={m['snmad']:.4f}  MAE={m['mae']:.3f}  out={m['out']:.1%}")


## Combined poster figure

Left 2×2 = reconstruction examples; middle = zero-shot k-NN; right = few-shot linear.

In [ ]:
POSTER_RC = {
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
}
mpl.rcParams.update(POSTER_RC)

ORIG_C, RECON_C = '#808080', '#006bff'       # recon: original grey / reconstructed blue
panel_ids = [146, 116, 184, 163]             # EDIT ME: objects in the 2x2 recon grid
lo, hi    = MIN_REDSHIFT, MAX_REDSHIFT
R = R_mask

fig = plt.figure(figsize=(18, 6))
outer = fig.add_gridspec(1, 2, width_ratios=[1.25, 2.0], wspace=0.13)
gl = outer[0].subgridspec(2, 2, wspace=0.0, hspace=0.0)
recon_axes = [fig.add_subplot(gl[0, 0]), fig.add_subplot(gl[0, 1]),
              fig.add_subplot(gl[1, 0]), fig.add_subplot(gl[1, 1])]
# The two redshift panels share one y-axis and butt together (no gap), like the 2x2.
gp = outer[1].subgridspec(1, 2, wspace=0.0)
ax_zs = fig.add_subplot(gp[0])
ax_fs = fig.add_subplot(gp[1], sharey=ax_zs)

# ---- Left: 2x2 masked-reconstruction examples ----
for k, (ax, idx) in enumerate(zip(recon_axes, panel_ids)):
    m = R["recon_mask"][idx].clone()
    L_used = m.shape[0]
    if EDGE_TRIM > 0:
        m[:EDGE_TRIM] = False
        m[L_used - EDGE_TRIM:] = False
    z  = R["redshift"][idx].item()
    w  = R["wave"][idx][m].numpy()
    si = np.argsort(w); w = w[si]
    orig  = R["flux_norm"][idx][m].numpy()[si]
    recon = R["recon_norm"][idx][m].numpy()[si]

    ax.plot(w, orig,  color=ORIG_C,  lw=2.6, alpha=0.95, label='Original')
    ax.plot(w, recon, color=RECON_C, lw=2.6, alpha=0.95, ls='--', label='Reconstructed')
    # emission lines in the OBSERVED frame: rest wavelength * (1+z)
    elo, ehi = w[0], w[-1]
    evis = sorted([(nm, wl*(1.0+z)) for nm, wl in EMISSION_LINES.items()
                   if elo <= wl*(1.0+z) <= ehi], key=lambda t: t[1])
    eylo, eyhi = ax.get_ylim(); eysp = eyhi - eylo
    eylv = [eyhi - 0.04*eysp, eyhi - 0.20*eysp]
    for ei, (enm, ewl) in enumerate(evis):
        ax.axvline(ewl, color='#999999', ls='--', lw=1.0, alpha=0.8)
        ax.text(ewl, eylv[ei % 2], enm, ha='center', va='top', fontsize=9,
                color='#333333', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.75, pad=1.5, edgecolor='none'))
    #ax.text(0.5, 0.06, f'#{idx}   z = {z:.3f}', transform=ax.transAxes,
    #        ha='center', va='bottom', fontsize=11, fontweight='bold', color=RECON_C,
    #        bbox=dict(facecolor='white', alpha=0.65, pad=2.0, edgecolor='none'))
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(width=1.6, labelsize=11)
    row, col = k // 2, k % 2
    if row == 1:
        ax.set_xlabel(r'$\lambda_{\rm obs}$  (µm)', fontsize=13)
    else:
        ax.tick_params(labelbottom=False)
    if col == 0:
        ax.set_ylabel('Norm. flux', fontsize=13)
        ax.set_xticks([1.0, 1.2, 1.4, 1.6, 1.8])   # drop 2.0 so it doesn't collide with the right panel's 1.0
    else:
        ax.tick_params(labelleft=False)
    if k == 3:
        leg = ax.legend(fontsize=10, loc='upper left', framealpha=0.85)
        leg.get_frame().set_linewidth(1.8); leg.get_frame().set_edgecolor('black')

# ---- Middle / right: redshift probes (shared y-axis, butted together) ----
def _panel(ax, y, yp, title, color, show_ylabel=True, legend=True):
    m  = metrics(y, yp)
    zl = np.linspace(lo, hi, 100)
    ax.scatter(y, yp, s=6, alpha=0.3, edgecolors='none', color=color)
    ax.fill_between(zl, zl - 0.15*(1+zl), zl + 0.15*(1+zl), alpha=0.12, color='gray',
                    label=r'$\pm0.15(1+z)$')
    #ax.plot([lo, hi], [lo, hi], '--', lw=2, color='black', alpha=0.4, label='1:1')
    ax.set_xlabel(r'$z_{\rm true}$')
    if show_ylabel:
        ax.set_ylabel(r'$z_{\rm pred}$')
    # title + metrics inside the panel, lower-right corner (no axes title, no `out`)
    ax.text(0.97, 0.03,
            rf"{title}" + "\n" +
            rf"$R^2$={m['r2']:.3f}   $\sigma_{{\rm NMAD}}$={m['snmad']:.4f}",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=13, color=color,
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, pad=3.0, edgecolor='none'))
    offset = 0.05
    ax.set_xlim(lo - offset, hi + offset); ax.set_ylim(lo - offset*4, hi + offset*4)
    if legend:
        ax.legend(fontsize=12, loc='upper left')
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(width=1.6)
    return m

m_zs = _panel(ax_zs, y_va, pred_zs,     f"Zero-shot kNN (k={KNN_K})", "darkorange")
m_fs = _panel(ax_fs, y_va, pred_lin, f"Linear Probe", "royalblue",
              show_ylabel=False, legend=False)
plt.setp(ax_fs.get_yticklabels(), visible=False)     # shared y -> label only on the left panel
ax_zs.set_xticks([1.0, 1.5, 2.0, 2.5])               # drop 3.0 so it doesn't collide with the right panel's 1.0
ax_fs.set_xticks([1.0, 1.5, 2.0, 2.5, 3.0])

fig.patch.set_alpha(0.0)
outp = os.path.join(POSTER_DIR, "poster_combined_recon_redshift")
fig.savefig(outp + ".png", transparent=False, dpi=300, bbox_inches='tight')
fig.savefig("/home/yacheng/nexus/ssl_outthere/paper/ssl_outthere_paper/spectrum_bench.png", transparent=False, dpi=300, bbox_inches='tight')
print(f"Saved → {outp}.png")
plt.show()